# RAG Evaluation

This notebook evaluates the retrieval and answer-generation behavior of the F1 Regulations RAG assistant.

The goal is to verify that:
- Relevant regulation passages are retrieved
- Generated answers are grounded in the retrieved context
- Source of the data is returned
- Unsupported questions are handled reasonably

## Environment Setup

Install the required dependencies and add the project root to the Python path so the notebook can import the reusable modules from `src/`.

In [1]:
%pip install -q -r ../requirements.txt

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

Note: you may need to restart the kernel to use updated packages.


## Initialize the RAG Pipeline

Load the source regulations, split them into chunks, build the vector store, and create the retriever used for evaluation.

In [2]:
from src import config
from src.integration import load_documents, split_documents
from src.retrieval import build_vector_store, create_retriever
from src.generation import generate_answer, format_sources

docs = load_documents(config.RAW_DATA_DIR)
splits = split_documents(docs)

vectorstore = build_vector_store(splits)
retriever = create_retriever(vectorstore)


def ask_question(question):
    '''
    This helper function is used for evaluation purposes.
    It sends a test question through the RAG pipeline and displays the question, answer, and source.
    '''
    retrieved_docs = retriever.invoke(question)

    answer = generate_answer(
        question,
        retrieved_docs
    )

    sources = format_sources(retrieved_docs)

    print(f"User Question: {question} \n \n Response: {answer} \n \n Source: {sources}")

c:\Users\thorp\Documents\GitHub\rag-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 15835.68it/s]


## Test Questions

These questions are to evaluate if the RAG can correctly locate a regulation passage and return an answer that corresponds to the question.

In [8]:
# Basic and direct question to test the assistants response

question = "What is the cost cap for an F1 team this year?"

ask_question(question)

User Question: What is the cost cap for an F1 team this year? 
 
 Response: According to Article D4.1.2 of the 2026 Formula 1: Financial Regulations (F1 Teams), the Cost Cap for an F1 team in a Full Year Financial Regulations Reporting Period where 24 Competitions or fewer take place is US Dollars 215,000,000 adjusted, if applicable, for Indexation. 
 
 Source: [{'source': 'financial_regulations-f1_teams.pdf', 'page': 8}, {'source': 'financial_regulations-f1_teams.pdf', 'page': 8}, {'source': 'financial_regulations-f1_teams.pdf', 'page': 8}, {'source': 'financial_regulations-f1_teams.pdf', 'page': 8}]


In [4]:
# Procedural question to check if the assistant can retrieve and summarize a series of rules rather than a single fact

question = "What happens after a Formula 1 race is suspended, and what steps are followed before the race can restart?"

ask_question(question)

User Question: What happens after a Formula 1 race is suspended, and what steps are followed before the race can restart? 
 
 Response: According to Article B5.14.2, if the Race is suspended, the duration of the suspension will be added to this two (2) hour period up to a maximum total Race duration of three (3) hours.

Before the race can restart, the following steps are followed:

1. The formation lap for the Race is started behind the Safety Car (Article B5.10).
2. The maximum total Race duration of three (3) hours will commence at the time the green lights on the start line are shown.
3. When the leader crosses the Line at the end of the lap following the lap during which the sum of these periods ended, provided this does not result in the scheduled number of laps being exceeded.

Note that the specific steps and procedures may vary depending on the circumstances of the suspension, but Article B5.14.2 provides a general outline for how the race can be restarted after a suspension. 

In [5]:
# This question checks retrieval in cases where the question is informal or unclear

question = "Constructors vs Drivers championship?"

ask_question(question)

User Question: Constructors vs Drivers championship? 
 
 Response: The Constructors' Championship and the Drivers' Championship are two separate championships in the FIA Formula One World Championship.

The Constructors' Championship is awarded to the team that scores the most points throughout the season, based on the results of all its drivers. (Article 4.9)

The Drivers' Championship is awarded to the driver who scores the most points throughout the season. (Article 4.10) 
 
 Source: [{'source': 'financial_regulations-pu_manufacturers.pdf', 'page': 40}, {'source': 'financial_regulations-pu_manufacturers.pdf', 'page': 40}, {'source': 'financial_regulations-pu_manufacturers.pdf', 'page': 40}, {'source': 'financial_regulations-pu_manufacturers.pdf', 'page': 40}]


In [6]:
# Question that is related to the subject of F1, but not actually described in the regulations

question = "Who is the GOAT of F1, and why is it Ayrton Senna?"

ask_question(question)

User Question: Who is the GOAT of F1, and why is it Ayrton Senna? 
 
 Response: I could not find that information in the indexed documents.

However, according to the provided context, there is no mention of who the Greatest of All Time (GOAT) is. The context only defines key terms such as "F1 Team", "FIA Super Licence", and "F1 Driver". 
 
 Source: [{'source': 'general_provisions.pdf', 'page': 48}, {'source': 'general_provisions.pdf', 'page': 48}, {'source': 'financial_regulations-f1_teams.pdf', 'page': 46}, {'source': 'financial_regulations-f1_teams.pdf', 'page': 46}]


In [7]:
# Unsupported question to check if the model avoids inventing an answer if the requested information is not in the data

question = "Can you explain how fourier transformations work?"

ask_question(question)

User Question: Can you explain how fourier transformations work? 
 
 Response: I could not find that information in the indexed documents. 
 
 Source: [{'source': 'technical_regulations.pdf', 'page': 203}, {'source': 'technical_regulations.pdf', 'page': 203}, {'source': 'technical_regulations.pdf', 'page': 203}, {'source': 'technical_regulations.pdf', 'page': 203}]
